# Benchmark: Fine-tuned Qwen2.5-Coder-7B (Magicoder)

This notebook performs the same benchmark as `src/code_generation.ipynb` but using the fine-tuned model `hoangnh39/qwen2.5-coder-7b-magicoder`.

In [3]:
!pip install -q datasets requests torch peft bitsandbytes transformers trl accelerate sentencepiece wandb matplotlib

## 1. Core Imports and Configuration

In [4]:
import math
import multiprocessing
import json
import re
import os
import time
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login
from datasets import load_dataset
import torch
try:
    from google.colab import userdata
    USE_COLAB = True
except ImportError:
    USE_COLAB = False

NUM_SAMPLES = 1 # Number of samples to generate per problem
K_VALUE = 1 # K for pass@k metric
RESULTS_DIR = "data"
os.makedirs(RESULTS_DIR, exist_ok=True)

BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-7B"
ADAPTER_ID = "hoangnh39/qwen2.5-coder-7b-magicoder"

# HF Login
if USE_COLAB:
    hf_token = userdata.get('HF_TOKEN')
else:
    # Fallback or manual token input
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(hf_token, add_to_git_credential=True)
else:
    print("Warning: HF_TOKEN not found. You might need to login manually.")

## 2. Prompt Templates

In [5]:
ZERO_SHOT_PROMPT = """You are an expert Python programmer. Write the implementation for the following problem.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly (e.g., from typing import List, Optional, etc.).
- Do not assume any helper functions exist — define everything you use.
- The solution must be self-contained and executable.

Problem:
{prompt}

Code:
"""

FEW_SHOT_PROMPT = """You are an expert Python programmer. You will be provided some examples, and then a problem to solve.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly (e.g., from typing import List, Optional, etc.).
- Do not assume any helper functions exist — define everything you use.
- The solution must be self-contained and executable.

Example 1:
Problem:
def add(a, b):
    \"\"\"Return the sum of a and b.\"\"\"

Code:
```python
def add(a, b):
    \"\"\"Return the sum of a and b.\"\"\"
    return a + b
```

Example 2:
Problem:
def is_palindrome(s: str) -> bool:
    \"\"\"Check if a string is a palindrome.\"\"\"

Code:
```python
def is_palindrome(s: str) -> bool:
    \"\"\"Check if a string is a palindrome.\"\"\"
    return s == s[::-1]
```

Problem:
{prompt}

Code:
"""

PROMPT_TEMPLATES = {
    "zero_shot": ZERO_SHOT_PROMPT,
    "few_shot": FEW_SHOT_PROMPT,
}

## 3. Helper Functions

In [6]:
def run_eval(code, test_code, queue):
    local_scope = {}
    try:
        exec(code, local_scope)
        exec(test_code, local_scope)
        queue.put(True)
    except Exception:
        queue.put(False)

def exec_code(code, test_code, timeout=5):
    queue = multiprocessing.Queue()
    p = multiprocessing.Process(target=run_eval, args=(code, test_code, queue))
    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        p.join()
        return False

    return queue.get() if not queue.empty() else False

def calculate_pass_at_k(n, c, k):
    if n - c < k: return 1.0
    if c == 0: return 0.0
    try:
        return 1.0 - (math.comb(n - c, k) / math.comb(n, k))
    except ValueError: return 0.0

def clean_code(generated_text):
    match = re.search(r"```python\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match: return match.group(1)
    match = re.search(r"```\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match: return match.group(1)
    return generated_text.strip()

## 4. Model Initialization

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading Base Model: {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f"Loading Adapter: {ADAPTER_ID}...")
model = PeftModel.from_pretrained(model, ADAPTER_ID)
model.eval()

class QwenHFGenerator:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def generate(self, prompt_template, task_prompt, num_samples=1):
        full_prompt = prompt_template.format(prompt=task_prompt)
        inputs = self.tokenizer(full_prompt, return_tensors="pt").to(self.model.device)

        samples = []
        for _ in range(num_samples):
            try:
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=512,
                        temperature=0.2,
                        top_p=0.95,
                        do_sample=True,
                        pad_token_id=self.tokenizer.eos_token_id
                    )
                prompt_length = inputs["input_ids"].shape[1]
                generated_ids = outputs[0][prompt_length:]
                generated_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
                samples.append(generated_text)
            except Exception as e:
                print(f"Error: {e}")
                samples.append("")
        return samples

generator = QwenHFGenerator(model, tokenizer)

Loading Base Model: Qwen/Qwen2.5-Coder-7B...


config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

Loading Adapter: hoangnh39/qwen2.5-coder-7b-magicoder...


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/80.8M [00:00<?, ?B/s]

## 5. Evaluation Loop

In [8]:
dataset = load_dataset("openai_humaneval", split="test")

for template_name, prompt_template in PROMPT_TEMPLATES.items():
    print(f"\n===== Evaluating Fine-tuned Model [{template_name}] =====")
    results = []
    total_pass_at_k = 0

    for item in tqdm(dataset):
        task_id = item["task_id"]
        prompt = item["prompt"]
        test_cases = item["test"]
        entry_point = item["entry_point"]

        start_time = time.time()
        raw_samples = generator.generate(prompt_template, prompt, num_samples=NUM_SAMPLES)
        time_taken = time.time() - start_time

        correct_count = 0
        cleaned_codes = []
        for raw_text in raw_samples:
            code = clean_code(raw_text)
            eval_code = code + "\n" + test_cases + f"\ncheck({entry_point})\n"
            passed = exec_code(code, eval_code, timeout=5)
            if passed: correct_count += 1
            cleaned_codes.append({"raw": raw_text, "extracted": code, "passed": passed})

        pass_at_k = calculate_pass_at_k(NUM_SAMPLES, correct_count, K_VALUE)
        total_pass_at_k += pass_at_k

        results.append({
            "task_id": task_id,
            "samples": cleaned_codes,
            "pass@k": pass_at_k,
            "time_taken": time_taken
        })

    avg_pass_at_k = total_pass_at_k / len(dataset)
    print(f"FT Model [{template_name}] - Avg pass@{K_VALUE}: {avg_pass_at_k:.4f}")

    output_file = os.path.join(RESULTS_DIR, f"results_finetuned_{template_name}.json")
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=4)
    print(f"Results saved to {output_file}")

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]


===== Evaluating Fine-tuned Model [zero_shot] =====


100%|██████████| 164/164 [51:08<00:00, 18.71s/it]


FT Model [zero_shot] - Avg pass@1: 0.7439
Results saved to data/results_finetuned_zero_shot.json

===== Evaluating Fine-tuned Model [few_shot] =====


100%|██████████| 164/164 [1:09:23<00:00, 25.39s/it]

FT Model [few_shot] - Avg pass@1: 0.7744
Results saved to data/results_finetuned_few_shot.json
